![logo](../../logoline_c3s.svg)

<div class="alert alert-block alert-warning">
Please note that this repository is used for development and review, so quality assessments should be considered work in progress until they are merged into the main branch
</div>

# Consistency between windstorm tracking algorithms – Duration

Production date: 2026-11-DD.

## 🌍 Use case: C3S Enhanced Windstorm Service (EWS)

## ❓ Quality assessment question
* **How do the tracking algorithms included in the _Windstorm tracks and footprints_ dataset (Hodges, TempestExtremes) compare in terms of windstorm duration?**

As part of the Copernicus Climate Change Service's (C3S) Enhanced Windstorm Service (EWS),
the [_Windstorm tracks and footprints derived from reanalysis over Europe between 1940 to present_](https://doi.org/10.24381/bf1f06a9) dataset
is intended to help users in the insurance sector
better understand the risk caused by European windstorms or extra-tropical cyclones (ETCs).
ETCs are large low-pressure systems in mid-latitude regions associated with high winds and heavy rain [[Walker+20](https://doi.org/10.1002/wea.3861)].
These windstorms can cause severe socio-economic damages, with costs in the millions of euros and fatalities ranging from individuals to thousands [[Garnier+18](https://doi.org/10.1016/j.coastaleng.2017.06.014)].

To generate this dataset,
two windstorm tracking algorithms
–
_Hodges_/TRACK [[Hoskins+02](https://doi.org/10.1175/1520-0469(2002)059%3C1041:NPOTNH%3E2.0.CO;2), [Hodges+99](https://doi.org/10.1175/1520-0493(1999)127<1362:ACFFT>2.0.CO;2), [Hodges+95](https://doi.org/10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.CO;2)]
and
_TempestExtremes_ [[Ullrich+21](https://doi.org/10.5194/gmd-14-5023-2021)]
–
were applied to historical meteorological data from ECMWF's fifth-generation reanalysis ERA5 [[Hersbach+20](https://doi.org/10.1002/qj.3803)] to detect windstorm tracks.
The resulting dataset contains three products, namely
the windstorm tracks found by both algorithms,
the associated footprint of each windstorm,
and
yearly summary indicators for administrative regions.

Because the tracking algorithms have different approaches for detecting windstorms,
with
Hodges using 3-hourly 850 hPa relative vorticity
and
TempestExtremes using 6-hourly mean sea level pressure (MSLP),
they are intended to be complementary to each another.
ETCs are highly complex and diverse systems
and
different tracking algorithms have different sensitivities, strengths, and limitations [[Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1)].
For example,
the MSLP approach generally represents low-frequency, large-scale features,
while vorticity represents high-frequency, small-scale features
[[Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1), [Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)]

Here,
we compare the outputs of both algorithms in terms of **duration**,
i.e.
when individual windstorm tracks are first/last detected and their total duration.
This comparison is performed as an internal consistency check,
rather than a validation using external data.
The aim is to provide guidance on the choice of algorithm and the consequences of using one over the other, or both at the same time.

## 📢 Quality assessment statement

```{admonition} These are the key outcomes of this assessment
:class: note
* The EWS _Windstorm tracks and footprints_ dataset contains around two thousand windstorms tracked with two different algorithms (Hodges and TempestExtremes). For 1940–2025, the distributions of windstorm durations from both algorithms peak around 2.8–3.9 days with some seasonal variation. Hodges tends to detect more, shorter windstorms that are not detected by TempestExtremes, skewing the Hodges distribution to the shorter end.
* For matched-up windstorm tracks, both algorithms find similar durations, with a median bias of 0 (interquartile range –0.25 to +0.25). There are some outliers with larger differences, up to several days.
* When analysing windstorm duration statistics, data from either tracking algorithm can be used with equal confidence, as they are internally consistent for matched-up windstorms. However, differences in sensitivity may skew statistics when only using a single algorithm.
```

## 📋 Methodology
This notebook examines the consistency in windstorm characteristics between the two tracking algorithms used in the [_Windstorm tracks and footprints derived from reanalysis over Europe between 1940 to present_](https://doi.org/10.24381/bf1f06a9) dataset.
Here,
we focus on the duration of windstorms as detected by each algorithm,
by investigating the individual windstorm tracks.

The analysis and results are organised in the following steps, which are detailed in the sections below:

**[](indicator_sis-european-wind-storm-reanalysis_consistency_q02:section-setup)**
* Import all required libraries.
* Define helper functions.

**[](indicator_sis-european-wind-storm-reanalysis_consistency_q02:section-duration)**
* Compare distribution of lifetimes for all windstorms between two algorithms.
* Compare distribution of lifetimes between matchups between two algorithms.

## 📈 Analysis and results

(indicator_sis-european-wind-storm-reanalysis_consistency_q02:section-setup)=
### 1. Code setup
```{note}
This notebook uses [earthkit](https://github.com/ecmwf/earthkit) for
downloading ([earthkit-data](https://github.com/ecmwf/earthkit-data))
and visualising ([earthkit-plots](https://github.com/ecmwf/earthkit-plots)) data.
Because earthkit is in active development, some functionality may change after this notebook is published.
If any part of the code stops functioning, please raise an issue on our GitHub repository so it can be fixed.
```

#### Import required libraries
In this section, we import all the relevant packages needed for running the notebook.

In [1]:
# Input / Output
import earthkit.data as ekd

# General data handling
import calendar
from functools import partial, reduce
from operator import and_ as bitwise_and
import numpy as np
# np.seterr(divide="ignore")  # Ignore divide-by-zero warnings
import pandas as pd

# Windstorm track matchups
import haversine as hs

# Visualisation
import earthkit.plots as ekp
from earthkit.plots.styles import Style
import matplotlib.pyplot as plt
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["figure.constrained_layout.use"] = True
plt.rcParams["figure.labelweight"] = "bold"
plt.rcParams["figure.titleweight"] = "bold"
plt.rcParams["legend.columnspacing"] = 1.5
plt.rcParams["legend.handletextpad"] = 0.5
from matplotlib.lines import Line2D
import cartopy
from cartopy import crs as ccrs
from IPython.display import HTML

# Visualisation in Jupyter book -- automatically ignored otherwise
try:
    from myst_nb import glue
except ImportError:
    glue = None

# Type hints
from typing import Any, Iterable, Optional
from earthkit.plots.geography.domains import Domain
AnyDomain = (Domain | str)
from pandas.io.formats.style import Styler as PandasStyler

#### Helper functions
This section defines some functions and variables used in the following analysis, allowing code cells in later sections to be shorter and ensuring consistency.

##### Data downloading & (pre-)processing

The following cell contains some pre-defined constants for convenience,
such as a list of variable names in the data:

In [2]:
# Data
TRACKING_ALGORITHMS = ["hodges", "tempest_extremes"]

# Organisation of data
COORDINATES         = ["latitude", "longitude"]

# Seasons
MONTHS              = calendar.month_name[1:]    # Named months of the year; [1:] drops an empty entry at the start
DJF                 = calendar.month_name[12:] + calendar.month_name[1:3]  # Northern Hemisphere winter: December-January-February
MAM                 = calendar.month_name[3:6]   # Northern Hemisphere spring: March-April-May
JJA                 = calendar.month_name[6:9]   # Northern Hemisphere summer: June-July-August
SON                 = calendar.month_name[9:12]  # Northern Hemisphere autumn: September-October-November
SEASONS             = {"DJF": DJF, "MAM": MAM, "JJA": JJA, "SON": SON,}
month2season        = {month: season  # Convert from month to season, e.g. "December": "DJF"
                       for season, months in SEASONS.items()
                       for month in months
                      }

The following cell provides some convenience functions for working with Pandas DataFrames,
such as manipulating their indices:

In [3]:
# Helper function: Add dummy index level to existing DataFrame
# (used here to allow certain functions to be re-used with different data)
def add_dummy_index(df: pd.DataFrame, name: str, value: Any) -> pd.DataFrame:
    """ Add a new index level that contains just a dummy value. """
    return pd.concat({value: df}, names=[name])

##### Statistics

The following functions calculate and display statistics on windstorm track duration:

In [4]:
# Track dates
def summarise_track_dates(data: pd.DataFrame) -> pd.DataFrame:
    """
    For each windstorm track in `data`, get the start/end dates and most important meteorological parameters.
    """
    # Sort by timestamp first
    data = data.sort_values(["algorithm", "id", "time"])

    # Summarise each track (algorithm-id combination)
    summary = data.groupby(["algorithm", "id"]).agg(  # Apply aggregation operations to each pair of algorithm-id values
                  time_start=("time", "first"),
                  time_end=("time", "last"),
                  latitude_start=("latitude", "first"),
                  latitude_end=("latitude", "last"),
                  longitude_start=("longitude", "first"),
                  longitude_end=("longitude", "last"),
                  fg10_max=("fg10", "max"),
                  msl_min=("msl", "min"),
              )

    return summary


def timestamp_to_season(time: pd.Series) -> pd.Series:
    """ Given a Series of timestamps, return a Series of the corresponding seasons (DJF ... SON). """
    return time.dt.month_name().replace(month2season)


# Track durations
def compute_track_durations(track_summary: pd.DataFrame) -> pd.DataFrame:
    """ Given a windstorm track summary, compute the duration [in days] of each track from the start/end dates. """
    # Make a copy to avoid overwriting the original
    data = track_summary.copy()

    # Remove columns that are not time / coordinate related
    keep_keys = ["time", "latitude", "longitude"]
    data = data[[col for col in data.columns if any(key in col for key in keep_keys)]]

    # Calculate duration in days
    data["duration"] = (data["time_end"] - data["time_start"]) / pd.Timedelta(days=1)

    # Add season column
    data["season"] = timestamp_to_season(data["time_start"])

    # Reorder columns
    data = data[["time_start", "time_end", "duration", "season", "latitude_start", "latitude_end", "longitude_start", "longitude_end"]]

    return data


# Summary statistics on track durations
def track_duration_statistics(track_summary: pd.DataFrame, *, col: str="duration") -> pd.DataFrame:
    """
    Compute summary statistics of windstorm track duration, by season and by algorithm.
    """
    # Make a copy to avoid overwriting the original
    data = track_summary.copy()

    # Calculate overall statistics
    stats_all = data.groupby(["algorithm",])[col].describe()
    stats_all = add_dummy_index(stats_all, "season", "All")  # Add "season: All" index to prepare for concatenation later

    # Calculate statistics by season
    stats_by_season = data.groupby(["season", "algorithm",])[col].describe()

    # Combine into one table
    stats = pd.concat([stats_all, stats_by_season])
    stats = stats.loc[["All", *SEASONS.keys()]]  # Order: All, DJF, ... SON

    # Adjust data types
    stats["count"] = stats["count"].astype(int)

    return stats


# Display track duration summary statistics in Jupyter Book-friendly style
def display_track_duration_statistics(data: pd.DataFrame) -> PandasStyler:
    """
    Display track duration more cleanly.
    Show count as int, everything else as float with 2 decimals.
    Colour all rows by the first index for easy comparison.
    """
    # Setup: Colour by element on the first index (e.g. by season)
    first = data.index.get_level_values(0).unique()
    first_colours = {index_value: "#f2f2f2" if i % 2 == 0 else "#ffffff"
                     for i, index_value in enumerate(first)}

    def colour_row(row):
        colour = first_colours[row.name[0]]
        return [f"background-color: {colour}"] * len(row)

    def colour_index(index):
        return [f"background-color: {first_colours[index_value]}"
                for index_value in data.index.get_level_values(0)]

    # Determine column position of "std"
    std_col = data.columns.get_loc("std")
    std_col_divider_style = [("border-right", "1px solid #999")]

    styled = (
        data.style
            .format(lambda x: f"{x:.2f}")                    # Show all columns with 2 decimals
            .format(lambda x: f"{x:.0f}", subset=["count"])  # Show count with 0 decimals (integer)
            .apply(colour_row, axis=1)  # Colour data  according to first index level
            .apply_index(               # Colour index according to first index level
                colour_index,
                axis=0, level=[0, 1],
            )
            # Add border between "std" and "min" columns to indicate difference between statistics and quantiles
            .set_table_styles([
                # Column heading
                {"selector": f"th.col_heading.col{std_col}", "props": std_col_divider_style,},
                # Index-name header row
                {"selector": f"th.blank.col{std_col}", "props": std_col_divider_style,},
                # Data cells
                {"selector": f"td.col{std_col}", "props": std_col_divider_style,},
            ], overwrite=False)
    )
    return styled

The following functions find and analyse track matchups following the method described in [[Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)]:

In [5]:
# Match / Filter by time
def find_overlapping_timestamps(data: pd.DataFrame, *, algorithms: Iterable[str]=TRACKING_ALGORITHMS) -> pd.DataFrame:
    """ Find timestamps that occur in both elements of `algorithms`, e.g. "hodges" and "tempest_extremes". """
    # Split dataset by tracking algorithm
    data_split = [data.loc[alg].reset_index(level="id") for alg in algorithms]

    # Find overlapping timestamps between the two algorithms for each storm ID
    time_overlap = pd.merge(*data_split,  # * splats the individual elements of data_split
                            on="time", how="inner",  # Find all points where the "time" column matches between algorithms
                            suffixes=[f"_{alg}" for alg in algorithms]  # Suffix with _hodges, _tempest_extremes
                           )
    return time_overlap

def match_by_time(data: pd.DataFrame, *, algorithms: Iterable[str]=TRACKING_ALGORITHMS) -> pd.DataFrame:
    """ Identify matching tracks based on temporal overlap, and calculate how much they overlap (% of length). """
    # Count overlapping time points for each track pair
    time_overlap = find_overlapping_timestamps(data, algorithms=algorithms)

    overlap_stats = (
        time_overlap
        .groupby([f"id_{alg}" for alg in algorithms])["time"]  # Loop through all matched pairs of (id_hodges, id_tempest_extremes)
        .agg(  # Calculate aggregated statistics
            length_overlap=pd.Series.nunique,  # Count the number of unique timestamps in each matched pair
            time_start="first",                # Get the first overlapping timestamp
        )
    ).reset_index()  # Turn (id_hodges, id_tempest_extremes) into columns
    
    overlap_stats = overlap_stats.rename(columns={"time": "length_overlap"})  # Rename column for clarity
    overlap_stats.index = overlap_stats.index.rename("id_matchup")            # Rename index for clarity

    # For each matching pair, get the lengths of the corresponding tracks in Hodges and TempestExtremes
    def process_individual_match_by_time(row_overlap: pd.Series) -> pd.Series:
        """ For one match-up row, get the individual track lengths. """
        # Get the original tracks for this ID in both algorithms
        ids_by_algorithm = {alg: row_overlap[f"id_{alg}"] for alg in algorithms}
    
        # Count how many timestamps this track has in both algorithms
        individual_tracks = {alg: data.loc[alg, ID] for alg, ID in ids_by_algorithm.items()}
        individual_lengths = {f"length_{alg}": track["time"].nunique() for alg, track in individual_tracks.items()}
    
        # Bundle into Pandas Series and return
        result = pd.Series(individual_lengths)
        return result

    individual_lengths = overlap_stats.apply(process_individual_match_by_time, axis=1)
    
    # Convert to DataFrame and filter matches
    matches_in_time = pd.concat([overlap_stats, individual_lengths], axis=1)

    # Determine how much overlap there is, for filtering later
    matches_in_time["length_mean"] = matches_in_time[[f"length_{alg}" for alg in algorithms]].mean(axis=1)
    matches_in_time["overlap_fraction"] = matches_in_time["length_overlap"] / matches_in_time["length_mean"]
    
    return matches_in_time

def filter_by_time_overlap(data: pd.DataFrame, *,
                           overlap_col: str="overlap_fraction", threshold: float=0.6) -> pd.DataFrame:
    """ Return only those elements of `data` where `overlap_col` >= threshold. Default is 0.6, from Hodges+03. """
    return data[data[overlap_col] >= threshold]


# Match / Filter by distance
def match_by_mean_distance(data: pd.DataFrame, *, existing_matchups: Optional[pd.DataFrame]=None,
                           algorithms: Iterable[str]=TRACKING_ALGORITHMS) -> pd.DataFrame:
    """
    Identify matching tracks based on temporal overlap, and calculate the mean distance between matched-up timestamps.
    `existing_matchups` can be a DataFrame with known match-ups, e.g. those obtained from matching and filtering on timestamp overlap.
        If provided, only matchups in `existing_matchups` will be considered.
    """
    # Find overlapping timestamps (i.e. timestamps that occur in both algorithms)
    # and calculate the distance between the locations provided by each algorithm
    time_overlap = find_overlapping_timestamps(data, algorithms=algorithms)
    coordinates_by_algorithm = [[f"{coordinate}_{algorithm}"
                                 for coordinate in COORDINATES]
                                for algorithm in algorithms]
    time_overlap_by_algorithm = [time_overlap[coords] for coords in coordinates_by_algorithm]
    time_overlap["mean_distance"] = hs.haversine_vector(*time_overlap_by_algorithm, unit=hs.Unit.KILOMETERS)

    # Calculate the mean distance for each match-up, across all overlapping timestamps
    distances_overlap = time_overlap.groupby([f"id_{alg}" for alg in algorithms])[["mean_distance"]].mean()
    distances_overlap = distances_overlap.reset_index()  # Turn (id_hodges, id_tempest_extremes) into columns
    distances_overlap.index = distances_overlap.index.rename("id_matchup")  # Rename index for clarity

    # If there were existing matchups (e.g. based on time), perform an inner join
    # This will filter out any matchups in `distances_overlap` that are not present in `existing_matchups`
    # (e.g. those filtered out on not having enough overlapping timestamps)
    if existing_matchups is not None:
        distances_overlap = (
            existing_matchups
            .reset_index()  # Move id_matchup to a column
            .merge(distances_overlap,  # Perform the inner join
                   how="inner",  # Find all points where the "time" column matches between algorithms
                   on=[f"id_{alg}" for alg in algorithms],
             )
            .set_index("id_matchup")  # Restore Matchup ID index
        )

    # Select best matches only -- sort by distance (closest first) and pick the first element for every storm
    distances_overlap = distances_overlap.sort_values("mean_distance")
    for alg in algorithms:  # Drop duplicates in Hodges, then in TempestExtremes (if provided in that order)
        distances_overlap = distances_overlap.drop_duplicates(subset=[f"id_{alg}"], keep="first")

    # Return result
    return distances_overlap

def filter_by_mean_distance(data: pd.DataFrame, *,
                            overlap_col: str="mean_distance", threshold: float=300.) -> pd.DataFrame:
    """ Return only those elements of `data` where `overlap_col` < threshold. Default is 300 km, from Hodges+03. """
    return data[data[overlap_col] < threshold]


# Display matchups
def display_track_matchups(data: pd.DataFrame, *, length: int=10) -> pd.DataFrame:
    """
    Display a table of track matchups (as a DataFrame, no HTML styling).
    Removes unwanted columns and applies rounding.
    `length` determines how many elements (at the start + at the end) are shown.
    """
    # Make copy to prevent overwriting data
    display_df = data.copy()  

    # Remove unwanted columns
    display_df = display_df.drop(columns=["length_overlap", "length_mean"])  # Redundant with existing columns

    # Apply number formatting
    display_df["overlap_fraction"] = display_df["overlap_fraction"].map(lambda x: f"{x:.0%}")
    display_df["mean_distance"]    = display_df["mean_distance"].map(   lambda x: f"{x:.0f}")

    return display_df

The following functions perform point-to-point comparisons of windstorms found by both tracking algorithms:

In [6]:
# Replace "length" columns with duration
def replace_length_with_duration(track_matchups: pd.DataFrame, track_summary: pd.DataFrame) -> pd.DataFrame:
    """
    Replace length_hodges and length_tempest_extremes columns [# of timestamps] in `track_matchups` with
        duration_hodges and duration_tempest_extremes [days] from `track_summary`.
    Returns a copy without modifying the original DataFrame.
    This may seem roundabout compared to just doing # timestamps * 0.25, but that approach
        does not account for the possibility of gaps in the track.
    """
    # Copy the original dataframe to avoid overwriting it
    track_matchups = track_matchups.copy()

    # For each algorithm, replace length with duration
    for algorithm in TRACKING_ALGORITHMS:
        # Add duration column
        track_matchups[f"duration_{algorithm}"] = track_matchups[f"id_{algorithm}"].map(
            track_summary.loc[algorithm, "duration"]
        )

        # Remove corresponding length column
        track_matchups = track_matchups.drop(columns=[f"length_{algorithm}"])

    return track_matchups


# Calculate point-by-point differences
def add_difference_columns(data: pd.DataFrame, *, col: str="duration") -> pd.DataFrame:
    """
    Add columns representing the difference in `col`:
        {col}_difference: x - y
        {col}_difference_relative: 2 * (x - y) / (x + y)
        {col}_difference_absolute: |x - y|
    """
    # Select data
    x, y = [data[f"{col}_{algorithm}"] for algorithm in TRACKING_ALGORITHMS]

    # Calculate differences
    difference          = (x - y).rename(f"{col}_difference")
    difference_relative = ((2 * difference) / (x + y)).rename(f"{col}_difference_relative")
    difference_absolute = difference.abs().rename(f"{col}_difference_absolute")

    # Add to original DataFrame (not in-place) and return
    data_updated = pd.concat([data, difference, difference_relative, difference_absolute,],
                             axis=1)
    return data_updated


# Summary statistics on track duration differences
def track_duration_difference_statistics(data: pd.DataFrame, *, col: str="duration") -> pd.DataFrame:
    """
    Compute summary statistics of windstorm track duration difference (between algorithms), by season.
    """
    # Make a copy to avoid overwriting the original
    data = data.copy()

    # Select only difference columns
    data = data.rename(columns={f"{col}_difference":          "Difference",
                                f"{col}_difference_relative": "Difference [%]",
                                f"{col}_difference_absolute": "Difference (absolute)",
                               },
                      )
    data = data[["Difference", "Difference [%]", "Difference (absolute)", "season",]]  # Select columns
    data["Difference [%]"] = data["Difference [%]"] * 100  # Convert to %

    # Calculate overall statistics
    stats_all = data.describe().T  # T to transpose, which is necessary because we have multiple columns
    stats_all = add_dummy_index(stats_all, "season", "All")  # Add "season: All" index to prepare for concatenation later

    # Calculate statistics by season
    stats_by_season = data.groupby("season").describe()
    stats_by_season = stats_by_season.stack(level=0, future_stack=True)  # Transpose
    # Note: `future_stack=True` will be deprecated in Pandas 3.0 and may cause errors in future.

    # Combine into one table
    stats = pd.concat([stats_all, stats_by_season])
    stats = stats.loc[["All", *SEASONS.keys()]]  # Order: All, DJF, ... SON

    # Adjust data types
    stats["count"] = stats["count"].astype(int)

    return stats

##### Visualisation

The following cell defines
the styles used for the different algorithms in line plots:

In [7]:
# General styles for consistency
# Colours come from Paul Tol's notes (https://sronpersonalpages.nl/~pault/)
ALGORITHM_COLOURS = {"hodges": "#997700", "tempest_extremes": "#6699CC",
                     "difference": "black", "ratio": "black", "matchup": "black",
                    }
ALGORITHM_MARKERS = {"hodges": "s", "tempest_extremes": "^",
                     "difference": "o", "ratio": "o", "matchup": "o",
                    }

The following cell contains some base helper functions (e.g. displaying in Jupyter Notebook or Jupyter Book style):

In [8]:
# Visualisation: Helper functions, general
def _glue_or_show(fig: plt.Figure, glue_label: Optional[str]=None) -> None:
    """
    If `glue` is available, glue the figure using the provided label.
    If not, display the figure in the notebook.
    """
    try:
        glue(glue_label, fig, display=False)
    except TypeError:
        plt.show()
    finally:
        plt.close()

def _add_textbox_to_subplots(text: str, *axs: plt.Axes | ekp.Subplot, right=False) -> None:
    """ Add a text box to each of the specified subplots. """
    # Get the plt.Axes for each ekp.Subplot
    axs = [subplot.ax if isinstance(subplot, ekp.Subplot) else subplot for subplot in axs]

    # Set up location
    x = 0.95 if right else 0.05
    horizontalalignment = "right" if right else "left"

    # Add the text
    for ax in axs:
        ax.text(x, 0.95, text, transform=ax.transAxes,
        horizontalalignment=horizontalalignment, verticalalignment="top",
        bbox={"facecolor": "white", "edgecolor": "black", "boxstyle": "round",
              "alpha": 1})

The following functions are also base helper functions, but specific to geospatial plots.
Administrative boundaries are downloaded from
the _Geographic Information System of the Commission_ (GISCO)
through earthkit-plots,
and are © EuroGeographics.

In [9]:
# Helper constants
DOMAIN_TRACKS = "North Atlantic"

# Visualisation: Helper functions for geospatial plots
def decorate_fig(fig: ekp.Figure, *, title: Optional[str]="") -> None:
    """ Decorate an earthkit figure with land, coastlines, etc. """
    fig.land()
    fig.borders(source="gisco", color="black")
    fig.coastlines(color="black")
    fig.gridlines(linestyle=plt.rcParams["grid.linestyle"])
    fig.title(title)

The following functions display the distribution of windstorm durations:

In [10]:
# Helper function: Setup for figure panels
def create_subplots_by_season(*, figsize=(9, 4), **kwargs) -> tuple[plt.Figure, dict[str, plt.Axes]]:
    """
    Helper function to create a gridded figure.
        Left half: One big panel for the overall plot ("All").
        Right half: 2x2 smaller panels for individual seasons ("DJF" ... "SON").
    """
    fig, axs = plt.subplot_mosaic(
        [["All", "All", "DJF", "MAM",],   # Top row: All (double width), winter, spring
         ["All", "All", "JJA", "SON",]],  # Bottom:  All (double width), summer, autumn
        gridspec_kw={"hspace": 0.1, "wspace": 0.1},  # Padding
        sharex=True, sharey=True,
        figsize=figsize,
        **kwargs,
    )
    return fig, axs


# Distribution of durations -- In each algorithm
def plot_duration_distribution(data: pd.DataFrame, *,
                               bins: Iterable[float]=np.arange(0, 22.1, 0.5),  # Days
                               title: Optional[str]="",
                               glue_label: Optional[str]=None,) -> None:
    """
    Plot probability distributions of storm durations from `data`.
    data["duration"] and `bins` are assumed to be in days.
    Big panel showing overall distribution, smaller panels per season.
    """
    # Setup: Figure
    fig, axs = create_subplots_by_season()

    # Plot distributions
    hist_kwargs = {"bins": bins, "density": True,  # Consistent settings
                   "alpha": 0.8, "histtype": "step", "linewidth": 2.5,
                  }
    for algorithm in TRACKING_ALGORITHMS:
        # Overall
        data.loc[algorithm, "duration"].hist(ax=axs["All"], **hist_kwargs,
                                             label=algorithm, color=ALGORITHM_COLOURS[algorithm])

        # Per season
        for season in SEASONS:
            data[data["season"] == season].loc[algorithm, "duration"].hist(ax=axs[season], **hist_kwargs,
                                                                           color=ALGORITHM_COLOURS[algorithm])

    # Axis settings
    axs["All"].set_xlim(bins[0], bins[-1])
    axs["All"].set_ylim(ymin=0)
    axs["All"].set_xticks(np.arange(0, 25, 5))

    # Labels and styling
    axs["All"].legend(loc="center right")
    for label, ax in axs.items():
        _add_textbox_to_subplots(label, ax, right=True)
    fig.supxlabel("Windstorm duration [days]")
    fig.supylabel("Density")
    fig.suptitle(title)

    # Show result
    _glue_or_show(fig, glue_label)


# Distribution of durations -- Difference between algorithms
def plot_duration_difference_hist(data: pd.DataFrame, *,
                                  bins: Iterable[float]=np.arange(-4.125, 4.126, 0.25),  # Days; one bin for each possible value
                                  title: Optional[str]="",
                                  glue_label: Optional[str]=None,) -> None:
    # Setup: Figure
    fig, axs = create_subplots_by_season()

    # Plot distributions
    hist_kwargs = {"bins": bins, "density": True,  # Consistent settings
                   "color": "black",
                  }

    # Overall
    data["duration_difference"].hist(ax=axs["All"], **hist_kwargs,)

    # Per season
    for season in SEASONS:
        data[data["season"] == season]["duration_difference"].hist(ax=axs[season], **hist_kwargs,)

    # Axis settings
    axs["All"].set_xlim(bins[0], bins[-1])
    axs["All"].set_ylim(ymin=0)

    # Labels and styling
    for label, ax in axs.items():
        _add_textbox_to_subplots(label, ax, right=True)
    fig.supxlabel("Δ Windstorm duration [days]")
    fig.supylabel("Density")
    fig.suptitle(title)

    # Show result
    _glue_or_show(fig, glue_label)


# Point-by-point (scatter / density) comparison of duration
def plot_duration_scatter(data: pd.DataFrame, *,
                          gridsize: Iterable[int]=(20, 20),  # Number of hexagons; more means finer but also noisier
                          title: Optional[str]="",
                          glue_label: Optional[str]=None,) -> None:
    # Setup: Figure
    fig, axs = create_subplots_by_season()

    # Plot values
    lims = (0, 22)
    scatter_kwargs = {"extent": (*lims, *lims), "gridsize": gridsize,
                      "mincnt": 1, "cmap": "cividis",
                     }

    # Overall
    algorithm0, algorithm1 = TRACKING_ALGORITHMS
    axs["All"].hexbin(data[f"duration_{algorithm0}"].values.ravel(),
                      data[f"duration_{algorithm1}"].values.ravel(),
                      **scatter_kwargs,
                     )

    # Per season
    for season in SEASONS:
        data_season = data[data["season"] == season]
        axs[season].hexbin(data_season[f"duration_{algorithm0}"].values.ravel(),
                           data_season[f"duration_{algorithm1}"].values.ravel(),
                           **scatter_kwargs,
                          )

    # Axis settings
    for ax in axs.values():
        # Equal aspect ratio
        ax.set_aspect("equal", adjustable="box")
        ax.set_yticks(ax.get_xticks())

        # Highlight diagonal
        ax.axline((0, 0), slope=1, color=plt.rcParams["grid.color"], linewidth=0.8, linestyle="-")

    # Labels and styling
    for label, ax in axs.items():
        _add_textbox_to_subplots(label, ax, right=True)
    fig.supxlabel(algorithm0)
    fig.supylabel(algorithm1)
    fig.suptitle(title)

    # Show result
    _glue_or_show(fig, glue_label)

The following functions display geospatial comparisons of individual windstorm tracks:

In [11]:
# Helper functions
def select_track_data_for_ids(track_data: pd.DataFrame, track_ids: dict[str, int]) -> dict[str, pd.DataFrame]:
    """ Get the tracks for each algorithm/ID pair in `track_ids` from the main table `track_data`. """
    # Find data
    tracks_selected = {alg: track_data.loc[alg, ID] for alg, ID in track_ids.items()}

    # Pre-process: Sort by time, reset index, and select variables
    tracks_selected = {alg: track.sort_values("time") for alg, track in tracks_selected.items()}
    tracks_selected = {alg: track.reset_index(drop=True).set_index("time") for alg, track in tracks_selected.items()}
    tracks_selected = {alg: track[["latitude", "longitude"]] for alg, track in tracks_selected.items()}

    return tracks_selected


def plot_track_geospatial(chart: ekp.Map, track: pd.DataFrame, algorithm: str, **kwargs) -> None:
    """
    Plot a windstorm track on a geospatial map, with scatter markers and a dashed line.
    Uses algorithm-specific colours and markers.
    """
    # Setup
    kwargs_shared = {"transform": ccrs.PlateCarree()} | kwargs
    
    # Plot individual points
    chart.ax.scatter(track["longitude"], track["latitude"],
                     color=ALGORITHM_COLOURS[algorithm], marker=ALGORITHM_MARKERS[algorithm],
                     edgecolors="black", s=50,
                     zorder=3,
                     **kwargs_shared)

    # Plot lines connecting points
    chart.ax.plot(track["longitude"], track["latitude"],
                  color=ALGORITHM_COLOURS[algorithm],
                  linestyle="--", linewidth=3,
                  zorder=3-0.1, alpha=0.7,
                  **kwargs_shared)


# Main function
def compare_tracks_geospatial(track_data: pd.DataFrame, track_ids: dict[str, int] | Iterable[dict[str, int]],
                              glue_label: Optional[str]=None) -> None:
    """
    Plot a geospatial comparison between storm tracks in `track_data` from two different tracking algorithms.
    `track_ids` is a Dictionary of track IDs, e.g. {'id_hodges': 1702, 'id_tempest_extremes': 787,}.
    """
    # Check if multiple track_id pairs are being provided or only one
    if isinstance(track_ids, dict):  # Single pair
        track_ids = [track_ids,]  # Create list of length 1, to iterate over

    # Create map
    n_pairs = len(track_ids)
    n_rows = n_pairs // 2 + n_pairs % 2  # One row for every 2 (= n_cols) track_id pairs, plus the remainder (+1 for odd numbers)
    fig = ekp.Figure(rows=n_rows, columns=2, figsize=(10, 1+5*n_rows))
    # (figsize is tailored for a 1x2 plot, may not look good for other combinations)

    # Loop over each pair and plot them in their own panel
    for id_pair in track_ids:
        # Remove "id_" prefix
        id_pair = {key.removeprefix("id_"): val for key, val in id_pair.items()}

        # Select desired tracks only
        id_pair = {key: val for key, val in id_pair.items() if val is not None}  # Filter out `None` values (dummies)
        tracks_selected = select_track_data_for_ids(track_data, id_pair)

        # Plot tracks
        chart = fig.add_map(domain=DOMAIN_TRACKS)
        for algorithm, track in tracks_selected.items():
            plot_track_geospatial(chart, track, algorithm)

        # Legend for algorithm lines
        handles = [Line2D([0], [0],
                          color=ALGORITHM_COLOURS[alg], linestyle="--", linewidth=3,
                          marker=ALGORITHM_MARKERS[alg], markersize=8, markerfacecolor=ALGORITHM_COLOURS[alg], markeredgecolor="black",
                          label=alg,)
                   for alg in tracks_selected]
        chart.ax.legend(handles=handles, loc="upper left")

        # Title
        title_ids = " – ".join(f"{alg} ID {ID}" for alg, ID in id_pair.items())
        chart.title(title_ids)

    # Decorate figure
    decorate_fig(fig, title="Storm track comparison")

    # Show result
    _glue_or_show(fig.fig, glue_label)

The following functions display comparisons of windstorm track duration:

(indicator_sis-european-wind-storm-reanalysis_consistency_q02:section-duration)=
### 2. Windstorm duration
We compare the windstorm tracks found by the Hodges and TempestExtremes algorithms.
The following cell defines the years for which we will download data;
by default, this is 1940–2025, but these years can be modified below when running the notebook yourself:

In [12]:
# Customisation
# Modify the range below to change the years considered
year_start, year_end = 1940, 2025

#### Download windstorm tracks
We download the windstorm tracks from the
[sis-european-wind-storm-reanalysis](https://doi.org/10.24381/bf1f06a9)
CDS entry.
This is a simple table which can be loaded into a Pandas [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).
For each algorithm (Hodges, TempestExtremes),
this table contains all timestamps with corresponding location (latitude, longitude) and some fields describing physical characteristics at that timestamp and location (see the [Product User Guide](https://confluence.ecmwf.int/x/jHaXGw#WindstormtracksandfootprintsderivedfromreanalysisoverEuropebetween1940topresent:ProductUserGuide-Windstormtrack) for details).

Note that the `id` column in this table is defined separately for each algorithm.
For example, the windstorm with `id` 42 in the Hodges list is likely _not_ the same as the windstorm with that `id` in the TempestExtremes list.

This notebook uses [earthkit-data](https://github.com/ecmwf/earthkit-data) to download files from the CDS.
If you intend to run this notebook multiple times, it is highly recommended that you [enable caching](https://earthkit-data.readthedocs.io/en/latest/concepts/caching.html) to prevent having to download the same files multiple times.
If you prefer not to use earthkit, the following requests can also be used with the [cdsapi module](https://cds.climate.copernicus.eu/how-to-api).
In either case (earthkit-data or cdsapi), it is required to set up a CDS account and API key as explained [on the CDS website](https://cds.climate.copernicus.eu/how-to-api).

In [13]:
# Define request for variables, based on CDS request form
dataset = "sis-european-wind-storm-reanalysis"

request = {
    "product": "windstorm_track",
    "variable": "all",
    "tracking_algorithm": ["hodges", "tempestextremes"],
    "event_aggregation": "single_event",
    "year":  [f"{year}"     for year  in range(year_start, year_end+1)],
    "month": [f"{month:02}" for month in range(1, 13)],
    "day":   [f"{day:02}"   for day   in range(1, 32)],
}

# Download data and load into Pandas
tracks = ekd.from_source("cds", dataset, request)
tracks = tracks.to_pandas()

# Pre-process: Use Pandas timestamps and set a custom index
tracks["time"] = pd.to_datetime(tracks["time"])
tracks = tracks.set_index(["algorithm", "id"])  # Reindex

# Display in notebook
tracks

time  latitude  longitude       fg10  \
algorithm        id                                                        
hodges           1   1940-02-22 00:00:00     63.25     -24.00  10.452867   
                 1   1940-02-22 06:00:00     63.50     -25.00  12.599429   
                 1   1940-02-22 12:00:00     62.75     -11.25   5.919555   
                 1   1940-02-22 18:00:00     63.00     -12.00   7.741296   
                 1   1940-02-23 00:00:00     62.75     -12.25   4.190840   
...                                  ...       ...        ...        ...   
tempest_extremes 871 2025-10-29 00:00:00     57.50      -7.50   7.256220   
                 871 2025-10-29 06:00:00     57.75      -6.50   5.611463   
                 871 2025-10-29 12:00:00     58.50      -2.25   7.861186   
                 871 2025-10-30 12:00:00     55.75      15.75   7.850687   
                 871 2025-10-30 18:00:00     56.50      19.00   5.625481   

                           lsm        msl  
algorithm        id                        
hodges           1    0.000000  98962.875  
                 1    0.000000  98905.190  
                 1    0.000000  98809.125  
                 1    0.000000  98755.560  
                 1    0.000000  98655.310  
...                        ...        ...  
tempest_extremes 871  0.203919  98960.750  
                 871  0.058715  98859.560  
                 871  0.000000  98899.120  
                 871  0.000000  98927.310  
                 871  0.000000  98920.690  

[34254 rows x 6 columns]

#### Calculate windstorm duration
We calculate windstorm duration as simply the time difference between the first and last detections of a given windstorm ID:

In [14]:
# Generate summary table
track_summary = summarise_track_dates(tracks)

# Compute track duration
track_summary = compute_track_durations(track_summary)

# Display subset of columns in notebook
columns_subset = [col for col in track_summary.columns 
                  if any(key in col for key in ["time", "season", "duration",])
                 ]
track_summary[columns_subset]

time_start            time_end  duration season
algorithm        id                                                          
hodges           1   1940-02-22 00:00:00 1940-02-24 12:00:00      2.50    DJF
                 2   1940-03-12 00:00:00 1940-03-15 06:00:00      3.25    MAM
                 3   1940-03-14 00:00:00 1940-03-15 18:00:00      1.75    MAM
                 4   1940-03-17 12:00:00 1940-03-20 12:00:00      3.00    MAM
                 5   1940-04-06 00:00:00 1940-04-08 12:00:00      2.50    MAM
...                                  ...                 ...       ...    ...
tempest_extremes 867 2025-06-26 12:00:00 2025-06-29 06:00:00      2.75    JJA
                 868 2025-08-07 18:00:00 2025-08-09 12:00:00      1.75    JJA
                 869 2025-10-03 00:00:00 2025-10-05 18:00:00      2.75    SON
                 870 2025-10-18 06:00:00 2025-10-20 18:00:00      2.50    SON
                 871 2025-10-27 12:00:00 2025-10-30 18:00:00      3.25    SON

[2770 rows x 4 columns]

#### Distribution comparison (all windstorms)
First,
we compare the overall distribution of windstorm durations between the two algorithms.
This is done statistically
in terms of
the sample mean, standard deviation,
and
quartiles (minimum, 25%, median or 50%, 75%, and maximum)
of the respective windstorm track duration distributions.
To account for potential seasonal differences,
this analysis is performed across
all windstorm tracks
and
individually for each of the four northern seasons
(DJF, MAM, JJA, SON).

In [15]:
# Calculate statistics per season
stats = track_duration_statistics(track_summary)

# Display in notebook
display_track_duration_statistics(stats)

Durations found by TempestExtremes tend to be longer than those found by Hodges.
For both algorithms,
the distribution has a strong peak at durations of 2–4 days with a short, fat tail towards longer durations
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-distribution-all>`),
with Hodges being more strongly skewed towards shorter durations.
This pattern is clearest in the overall, DJF (winter), and MAM (spring) distributions,
while the JJA (summer) and SON (autumn) distributions appear more similar both visually and statistically.

There are two potential explanations for these differences.
The first is that the algorithms find different durations for the same windstorm tracks,
i.e. one detects the same storm earlier or for a longer time than the other.
The second is that one algorithm tends to find more shorter storms that the other does not detect at all.
To distinguish between these two effects,
we must look only at those windstorms detected by both algorithms.

In [16]:
# Histogram of track durations
plot_duration_distribution(track_summary,
                           title="Windstorm duration distribution (all windstorms)",
                           glue_label="indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-distribution-all")

```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-distribution-all
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-distribution-all"

Distributions of windstorm duration for windstorm tracks
found by the Hodges and TempestExtremes algorithms,
in 12-hour bins.
The probability density is relative to the total number of windstorm tracks in each algorithm individually.
```

#### Distribution comparison (matchups only)
Next,
we compare the behaviour of the two algorithms for windstorm tracks found by both.
This requires applying a track-to-track matchup method,
for which we use the one described in [[Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)],
which works as follows:
* For each pair of tracks (Hodges–TempestExtremes), find the number of matching timestamps.
* Keep only those pairs with ≥60% (relative to the mean track length between the two algorithms) overlap.
* For each remaining pair of tracks, compute the mean distances between matched-up timestamps.
* Keep only those pairs where the mean distance is <300 km.

In [17]:
# Find matchups in time and keep only those that overlap >= 60%
# To change this threshold, change the value of `threshold=0.6` to your desired value
track_matchups_time = match_by_time(tracks)
track_matchups_time = filter_by_time_overlap(track_matchups_time, threshold=0.6)

# Calculate mean distances between matchups and keep only those with mean distances < 300 km
# To change this threshold, change the value of `threshold=300` to your desired value
track_matchups_distance = match_by_mean_distance(tracks, existing_matchups=track_matchups_time)
track_matchups_distance = filter_by_mean_distance(track_matchups_distance, threshold=300)

# Finalise (better variable name)
matchups = track_matchups_distance
del track_matchups_time, track_matchups_distance  # Delete variables to avoid accidentally using the wrong one later

# Display result
display_track_matchups(matchups)

,id_hodges,id_tempest_extremes,time_start,length_hodges,length_tempest_extremes,overlap_fraction,mean_distance
id_matchup,,,,,,,
1115,1702,787,2017-10-15 06:00:00,9,9,100%,0
1087,1666,770,2016-01-01 00:00:00,6,8,86%,0
576,887,426,1982-11-08 18:00:00,12,12,100%,0
383,602,291,1970-05-20 00:00:00,8,15,61%,0
380,598,287,1970-02-11 18:00:00,8,8,100%,0
...,...,...,...,...,...,...,...
331,508,250,1965-12-30 06:00:00,18,25,74%,274
455,708,344,1975-03-01 06:00:00,14,20,76%,277
630,955,462,1985-04-11 12:00:00,11,19,67%,280


In [18]:
# Select only those tracks in the original DataFrame whose ID occurs in a matchup
matchups_tracks = pd.concat([
    tracks.loc[(algorithm, matchups[f"id_{algorithm}"]), :]
    for algorithm in TRACKING_ALGORITHMS
])

# Generate summary table
matchups_summary = summarise_track_dates(matchups_tracks)

# Compute track duration
matchups_summary = compute_track_durations(matchups_summary)

# Display in notebook
matchups_summary

time_start            time_end  duration season  \
algorithm        id                                                             
hodges           4   1940-03-17 12:00:00 1940-03-20 12:00:00      3.00    MAM   
                 8   1940-09-15 12:00:00 1940-09-18 18:00:00      3.25    SON   
                 12  1940-11-01 18:00:00 1940-11-05 00:00:00      3.25    SON   
                 16  1941-02-14 18:00:00 1941-02-20 18:00:00      6.00    DJF   
                 21  1941-08-27 12:00:00 1941-08-28 18:00:00      1.25    JJA   
...                                  ...                 ...       ...    ...   
tempest_extremes 866 2025-03-06 12:00:00 2025-03-08 06:00:00      1.75    MAM   
                 867 2025-06-26 12:00:00 2025-06-29 06:00:00      2.75    JJA   
                 868 2025-08-07 18:00:00 2025-08-09 12:00:00      1.75    JJA   
                 869 2025-10-03 00:00:00 2025-10-05 18:00:00      2.75    SON   
                 871 2025-10-27 12:00:00 2025-10-30 18:00:00      3.25    SON   

                      latitude_start  latitude_end  longitude_start  \
algorithm        id                                                   
hodges           4             48.25         56.75           -29.25   
                 8             58.50         60.00           -36.00   
                 12            44.75         52.25           -42.75   
                 16            35.25         54.50           -23.75   
                 21            55.00         60.25           -14.25   
...                              ...           ...              ...   
tempest_extremes 866           56.75         66.75           -11.75   
                 867           52.75         67.00           -41.75   
                 868           62.00         62.25           -24.75   
                 869           49.75         57.25           -24.75   
                 871           58.75         56.50           -22.75   

                      longitude_end  
algorithm        id                  
hodges           4            -4.00  
                 8             9.75  
                 12           14.50  
                 16            3.75  
                 21           -1.75  
...                             ...  
tempest_extremes 866          19.25  
                 867           3.50  
                 868          -4.50  
                 869          11.00  
                 871          19.00  

[1298 rows x 8 columns]

We calculate the same statistics as before,
now only considering matched-up storms:

In [19]:
# Calculate statistics per season
stats_matchups = track_duration_statistics(matchups_summary)

# Display in notebook
display_track_duration_statistics(stats_matchups)

Note that there is a small discrepancy in storm count by season,
due to some storms occurring on the cusp of a season and being first detected right before vs. right after the start of said season by the respective algorithms.

The statistics for matched-up windstorm track durations are strikingly similar between the two algorithms.
Some differences between the distributions remain
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-distribution-matchup>`),
such as Hodges tending to produce shorter durations,
particularly in winter,
but these differences are much smaller than before.

We can hence conclude that the difference in overall distribution seen above was caused primarily by differences in sensitivity to short windstorms
(further discussed in a
[separate quality assessment](./indicator_sis-european-wind-storm-reanalysis_consistency_q01))
rather than by differences in how long the two algorithms track the same windstorm.
This is illustrated by two examples
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-track-geospatial-comparison>`),
one being a storm from 1965 that the two algorithms first detect at different locations and timestamps,
the other (2017's [Storm Ophelia](https://en.wikipedia.org/wiki/Hurricane_Ophelia_(2017))) being tracked identically by both algorithms.

In [20]:
# Histogram of track durations
plot_duration_distribution(matchups_summary,
                           title="Windstorm duration distribution (matchups only)",
                           glue_label="indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-distribution-matchup")

In [21]:
# Specify the storm tracks you want to compare using the respective Hodges and TempestExtremes IDs as below.
# These can be derived by manual inspection of the comparison table.
storm_example   = {"id_hodges":  508, "id_tempest_extremes":  250,}  # From matchup table, December 1965
storm_ophelia   = {"id_hodges": 1702, "id_tempest_extremes":  787,}  # Storm Ophelia,      October  2017

storms_to_plot = [storm_example, storm_ophelia]

# Plot
compare_tracks_geospatial(tracks, storms_to_plot,
                          glue_label="indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-track-geospatial-comparison",
                         )

::::{tab-set}
:::{tab-item} Distribution
:sync: distribution
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-distribution-matchup
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-distribution-matchup"

Distributions of windstorm duration for windstorm tracks
found by both the Hodges and TempestExtremes algorithms (matched-up),
in 12-hour bins.
The probability density is relative to the number of matched-up windstorms.
```
:::
:::{tab-item} Individual track comparisons
:sync: tracks
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-track-geospatial-comparison
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-track-geospatial-comparison"

Geospatial comparison of storm tracks from each algorithm
(Hodges, TempestExtremes)
for individual windstorm events.
```
:::
::::

#### Point-by-point comparison
Lastly,
we perform a point-by-point comparison of individual matched-up windstorm tracks.

In [22]:
# Add "season" column based on matched-up time_start
matchups["season"] = timestamp_to_season(matchups["time_start"])

# Add durations to track matchups
matchups = replace_length_with_duration(matchups, matchups_summary)

# Calculate differences
matchups = add_difference_columns(matchups)

# Display subset of columns in notebook
columns_subset = [col for col in matchups.columns 
                  if any(key in col for key in ["id", "season", "fraction", "duration"])
                 ]
matchups[columns_subset]

,id_hodges,id_tempest_extremes,overlap_fraction,season,duration_hodges,duration_tempest_extremes,duration_difference,duration_difference_relative,duration_difference_absolute
id_matchup,,,,,,,,,
1115,1702,787,1.000000,SON,2.00,2.00,0.00,0.000000,0.00
1087,1666,770,0.857143,DJF,1.25,1.75,-0.50,-0.333333,0.50
576,887,426,1.000000,SON,2.75,2.75,0.00,0.000000,0.00
383,602,291,0.608696,MAM,2.00,5.50,-3.50,-0.933333,3.50
380,598,287,1.000000,DJF,1.75,1.75,0.00,0.000000,0.00
...,...,...,...,...,...,...,...,...,...
331,508,250,0.744186,DJF,4.25,6.25,-2.00,-0.380952,2.00
455,708,344,0.764706,MAM,3.25,5.00,-1.75,-0.424242,1.75
630,955,462,0.666667,MAM,2.50,4.75,-2.25,-0.620690,2.25


We quantify the difference using the same statistics as before
(mean, standard deviation, quartiles),
for the difference Hodges – TempestExtremes
including
the absolute value of the difference
and
the difference in % relative to the mean between the two algorithms:

In [23]:
# Calculate statistics on differences
difference_stats = track_duration_difference_statistics(matchups)

# Display in notebook
display_track_duration_statistics(difference_stats)

Overall and for each individual season,
the mean and median differences are (close to) 0 days,
indicating that there is no consistent bias between the two algorithms.
The distribution of differences is sharply peaked around 0
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-difference-hist>`),
with thin tails beyond ±0.75 days.
In a scatter plot comparing durations from the two algorithms,
most points fall along the diagonal line
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-difference-scatter>`),
with a small number of outliers.

Since most windstorm tracks have durations of 2–4 days
(see above),
even a difference of one timestamp (0.25 days) can correspond to a relative difference of 6.3–12.5%
as seen in the table.
As such, these value should be considered as relatively small differences.

An important caveat to this analysis is that the matchup algorithm selectively filters for pairs of windstorm tracks
(from the two algorithms)
that overlap by ≥60% relative to their mean track length [[Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)].
This does not correspond directly to a threshold in duration difference,
but certainly biases the sample of matchups towards those with better agreements.
However,
this is essentially inevitable when performing matchups.
Users interested in investigating this bias may re-run this notebook with different settings for the matchup algorithm,
as explained above in the relevant code cells.

In conclusion,
where windstorm tracks can be matched up between the two algorithms included in this dataset,
the resulting durations agree well
(interquartile range of –0.25 to +0.25 days difference)
without systematic biases.

In [24]:
# Histogram comparison
plot_duration_difference_hist(matchups,
                              title="Point-by-point difference in windstorm duration",
                              glue_label="indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-difference-hist")

# Scatter comparison
plot_duration_scatter(matchups,
                      title="Point-by-point comparison between algorithms",
                      glue_label="indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-difference-scatter")

::::{tab-set}
:::{tab-item} Histogram
:sync: hist
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-difference-hist
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-difference-hist"

Point-by-point difference between track durations derived from the Hodges and TempestExtremes algorithms
(Hodges – TempestExtremes).
Positive values indicate Hodges found a longer duration,
negative TempestExtremes.
The density is relative to the total number of windstorm track matchups.
```
:::
:::{tab-item} Scatter
:sync: scatter
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-difference-scatter
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q02_fig-difference-scatter"

Point-by-point comparison between track durations derived from the Hodges and TempestExtremes algorithms.
Colour indicates the number of points within that hexagon,
blue being low and yellow high.
```
:::
::::

## ℹ️ If you want to know more

### Key resources
The CDS catalogue entries for the data used were:
* Windstorm tracks and footprints derived from reanalysis over Europe between 1940 to present: [sis-european-wind-storm-reanalysis](https://doi.org/10.24381/bf1f06a9)
  * [](./indicator_sis-european-wind-storm-reanalysis_consistency_q01)
  * **Consistency between windstorm tracking algorithms – Duration**

More about windstorm tracking algorithms:
* [IMILAST: A Community Effort to Intercompare Extratropical Cyclone Detection and Tracking Algorithms](https://doi.org/10.1175/BAMS-D-11-00154.1)
* [The numerous approaches to tracking extratropical cyclones and the challenges they present](https://doi.org/10.1002/wea.3861)
* Hodges / TRACK
  * [Feature Tracking on the Unit Sphere](https://doi.org/10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.CO;2)
  * [Adaptive Constraints for Feature Tracking](https://doi.org/10.1175/1520-0493(1999)127<1362:ACFFT>2.0.CO;2)
  * [New Perspectives on the Northern Hemisphere Winter Storm Tracks](https://doi.org/10.1175/1520-0469(2002)059<1041:NPOTNH>2.0.CO;2)
* TempestExtremes
  * [TempestExtremes v2.1: a community framework for feature detection, tracking, and analysis in large datasets](https://doi.org/10.5194/gmd-14-5023-2021)

Code libraries used:
* [GeoPandas](https://geopandas.org/en/stable/)
* [Haversine](https://github.com/mapado/haversine)
* [earthkit](https://earthkit.ecmwf.int/)
  * [earthkit-data](https://github.com/ecmwf/earthkit-data)
  * [earthkit-plots](https://github.com/ecmwf/earthkit-plots)

### References
[[Garnier+18](https://doi.org/10.1016/j.coastaleng.2017.06.014)] E. Garnier, P. Ciavola, T. Spencer, O. Ferreira, C. Armaroli, and A. McIvor, ‘Historical analysis of storm events: Case studies in France, England, Portugal and Italy’, Coastal Engineering, vol. 134, pp. 10–23, Apr. 2018, doi: 10.1016/j.coastaleng.2017.06.014.

[[Hersbach+20](https://doi.org/10.1002/qj.3803)] H. Hersbach et al., ‘The ERA5 global reanalysis’, Quarterly Journal of the Royal Meteorological Society, vol. 146, no. 730, pp. 1999–2049, May 2020, doi: 10.1002/qj.3803.

[[Hodges+95](https://doi.org/10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.CO;2)] K. I. Hodges, ‘Feature Tracking on the Unit Sphere’, Monthly Weather Review, vol. 123, no. 12, pp. 3458–3465, Dec. 1995, doi: 10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.&ZeroWidthSpace;CO;2.

[[Hodges+99](https://doi.org/10.1175/1520-0493(1999)127<1362:ACFFT>2.0.CO;2)] K. I. Hodges, ‘Adaptive Constraints for Feature Tracking’, Monthly Weather Review, vol. 127, no. 6, pp. 1362–1373, Jun. 1999, doi: 10.1175/1520-0493(1999)127<1362:ACFFT>2.0.&ZeroWidthSpace;CO;2.

[[Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)] K. I. Hodges, B. J. Hoskins, J. Boyle, and C. Thorncroft, ‘A Comparison of Recent Reanalysis Datasets Using Objective Feature Tracking: Storm Tracks and Tropical Easterly Waves’, Monthly Weather Review, vol. 131, no. 9, pp. 2012–2037, Sep. 2003, doi: 10.1175/1520-0493(2003)131<2012:ACORRD>2.0.&ZeroWidthSpace;CO;2.

[[Hoskins+02](https://doi.org/10.1175/1520-0469(2002)059<1041:NPOTNH>2.0.CO;2)] B. J. Hoskins and K. I. Hodges, ‘New Perspectives on the Northern Hemisphere Winter Storm Tracks’, Journal of the Atmospheric Sciences, vol. 59, no. 6, pp. 1041–1061, Mar. 2002, doi: 10.1175/1520-0469(2002)059<1041:NPOTNH>2.0.&ZeroWidthSpace;CO;2.

[[Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1)] U. Neu et al., ‘IMILAST: A Community Effort to Intercompare Extratropical Cyclone Detection and Tracking Algorithms’, Bulletin of the American Meteorological Society, vol. 94, no. 4, pp. 529–547, Apr. 2013, doi: 10.1175/BAMS-D-11-00154.1.

[[Ullrich+21](https://doi.org/10.5194/gmd-14-5023-2021)] P. A. Ullrich, C. M. Zarzycki, E. E. McClenny, M. C. Pinheiro, A. M. Stansfield, and K. A. Reed, ‘TempestExtremes v2.1: a community framework for feature detection, tracking, and analysis in large datasets’, Geoscientific Model Development, vol. 14, no. 8, pp. 5023–5048, Aug. 2021, doi: 10.5194/gmd-14-5023-2021.

[[Walker+20](https://doi.org/10.1002/wea.3861)] E. Walker, D. Mitchell, and W. Seviour, ‘The numerous approaches to tracking extratropical cyclones and the challenges they present’, Weather, vol. 75, no. 11, pp. 336–341, Nov. 2020, doi: 10.1002/wea.3861.